# 10 — FastAPI + Demo Application

This stage exposes the completed TalentMatch AI ranking system through an application layer.

The application reuses the final ranking results produced in Notebook 07.

It provides:

- Job selection
- Candidate ranking
- Final match percentage
- TF-IDF similarity
- Word2Vec similarity
- Skill coverage
- Matched skills
- Missing skills
- FastAPI endpoint
- Streamlit demonstration interface

No model is retrained in this stage.

## 1. Application Architecture

The application uses the existing final ranking pipeline as its backend data source.

Training and model development were completed in the previous notebooks.

The application layer is responsible only for exposing the existing results through an API and user interface.

In [1]:
import pandas as pd

FINAL_RESULTS_PATH = r"D:\Instant AI\AI\Graduation Project\My Project\Data\final_matching_results.csv"

df = pd.read_csv(FINAL_RESULTS_PATH)

print("Final ranking data loaded successfully.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Final ranking data loaded successfully.
Shape: (9544, 46)

Columns:
['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'skills_required', 'matched_score', 'candidate_text', 'job_text', 'candidate_text_length', 'job_text_length', 'split', 'tfidf_similarity', 'absolute_error', 'word2vec_similarity', 'skill_coverage_percentage', 'skill_coverage_score', 'matched_skills', 'missing_skills', 'candidate_record_id', 'tfidf_scaled', 'word2vec_scaled', 'final_match_score', 'final_ma

## 2. Validate the Application Data

The application requires the final ranking signals generated by the previous stages.

In [2]:
required_columns = [
    "candidate_record_id",
    "job_position_name",
    "tfidf_similarity",
    "word2vec_similarity",
    "skill_coverage_percentage",
    "skill_coverage_score",
    "matched_skills",
    "missing_skills",
    "final_match_percentage"
]

missing_required = [
    column
    for column in required_columns
    if column not in df.columns
]

print("Missing required columns:", missing_required)

if not missing_required:
    print("Application data validation passed.")
else:
    print("Application data validation failed.")

Missing required columns: []
Application data validation passed.


## 3. Prepare the Ranking Data

The final matching dataset already contains the similarity scores,
skill coverage score, and final match percentage.

These existing results will be used directly by the application.
No additional model training or matching calculations are performed here.

In [3]:
application_columns = [
    "candidate_record_id",
    "job_position_name",
    "tfidf_similarity",
    "word2vec_similarity",
    "skill_coverage_percentage",
    "skill_coverage_score",
    "matched_skills",
    "missing_skills",
    "final_match_percentage"
]

application_df = df[application_columns].copy()

print("Application dataset prepared successfully.")
print("Shape:", application_df.shape)

application_df.head()

Application dataset prepared successfully.
Shape: (9544, 9)


,candidate_record_id,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_percentage,skill_coverage_score,matched_skills,missing_skills,final_match_percentage
0,1,senior software engineer,0.003533,0.692982,0.000000,0.000000,[],"['documentation', 'software engineer', 'techni...",18.790379
1,2,machine learning (ml) engineer,0.056484,0.834400,37.878788,0.378788,"['data analysis', 'machine learning', 'machine...","['application development', 'functional', 'lea...",68.939394
2,3,"executive/ senior executive- trade marketing, ...",0.000000,0.473196,0.000000,0.000000,[],"['business administration', 'excel', 'marketin...",0.000000
3,4,business development executive,0.012055,0.681219,20.000000,0.200000,['fast'],"['business development', 'quality', 'reliable'...",32.519589
4,5,senior ios engineer,0.004766,0.549506,0.000000,0.000000,[],"['api', 'application development', 'integratio...",6.641692


## 4. Create the Ranking Function

The final match percentage was already calculated in the previous ranking stage.

This application only filters candidates for the selected job position and orders them by the existing final match percentage.

In [4]:
def rank_candidates(job_position, top_k=5):

    results = application_df[
        application_df["job_position_name"] == job_position
    ].copy()

    results = results.sort_values(
        "final_match_percentage",
        ascending=False
    )

    return results.head(top_k)

## 5. Test Candidate Ranking

A sample job position is selected from the existing dataset.

The system returns the highest-ranked candidates according to the final match percentage.

In [5]:
sample_job = application_df[
    "job_position_name"
].dropna().iloc[0]

ranked_results = rank_candidates(
    sample_job,
    top_k=5
)

print("Selected job:", sample_job)
print("\nTop candidates/results:")

ranked_results[
    [
        "job_position_name",
        "tfidf_similarity",
        "word2vec_similarity",
        "skill_coverage_percentage",
        "skill_coverage_score",
        "final_match_percentage"
    ]
]

Selected job: senior software engineer

Top candidates/results:


,job_position_name,tfidf_similarity,word2vec_similarity,skill_coverage_percentage,skill_coverage_score,final_match_percentage
3116,senior software engineer,0.046436,0.745629,87.5,0.875,90.269566
6544,senior software engineer,0.064307,0.791068,75.0,0.750,87.500000
1470,senior software engineer,0.043360,0.764839,75.0,0.750,85.069664
7585,senior software engineer,0.034158,0.787433,75.0,0.750,81.667085
9106,senior software engineer,0.030833,0.721862,87.5,0.875,80.444138


## 6. Prepare API Data

The API will return the ranking results as structured JSON.

The response includes the final score and the component scores used to explain the ranking.

In [6]:
api_results = ranked_results[
    [
        "job_position_name",
        "tfidf_similarity",
        "word2vec_similarity",
        "skill_coverage_percentage",
        "skill_coverage_score",
        "final_match_percentage"
    ]
].copy()

api_results = api_results.to_dict(
    orient="records"
)

print("API response prepared successfully.")
print(api_results)

API response prepared successfully.
[{'job_position_name': 'senior software engineer', 'tfidf_similarity': 0.0464357051762903, 'word2vec_similarity': 0.7456287, 'skill_coverage_percentage': 87.5, 'skill_coverage_score': 0.875, 'final_match_percentage': 90.26956628631076}, {'job_position_name': 'senior software engineer', 'tfidf_similarity': 0.0643071018722869, 'word2vec_similarity': 0.79106766, 'skill_coverage_percentage': 75.0, 'skill_coverage_score': 0.75, 'final_match_percentage': 87.5}, {'job_position_name': 'senior software engineer', 'tfidf_similarity': 0.04335990184835, 'word2vec_similarity': 0.76483923, 'skill_coverage_percentage': 75.0, 'skill_coverage_score': 0.75, 'final_match_percentage': 85.06966393299336}, {'job_position_name': 'senior software engineer', 'tfidf_similarity': 0.0341575925973382, 'word2vec_similarity': 0.7874326, 'skill_coverage_percentage': 75.0, 'skill_coverage_score': 0.75, 'final_match_percentage': 81.667085243191}, {'job_position_name': 'senior softwar

## 7. Create the FastAPI Backend

The API provides a job-to-candidate ranking endpoint.

It uses the completed final matching dataset and does not retrain or recalculate the NLP models.

In [7]:
import sys
from pathlib import Path

APP_DIR = Path(
    r"D:\Instant AI\AI\Graduation Project\My Project\app\10_application"
)

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from api import (
    app,
    RankingRequest,
    rank_job,
    rank_candidates
)

print("FastAPI application loaded successfully.")

FastAPI application loaded successfully.


## 8. Test the API Logic

Before running the API server, the ranking function is tested using the same structure expected by the API request.

In [8]:
test_request = RankingRequest(
    job_position_name=sample_job,
    top_k=3
)

test_response = rank_job(
    test_request
)

print(test_response)

{'job_position_name': 'senior software engineer', 'number_of_results': 3, 'results': [{'candidate_record_id': 3117, 'job_position_name': 'senior software engineer', 'job_text': 'senior software engineer b.sc in computer science & engineering from a reputed university. at least 1 year technical support troubleshooting collaboration documentation system monitoring software deployment training & mentorship industry trends field visits', 'tfidf_similarity': 0.0464357051762903, 'word2vec_similarity': 0.7456287, 'skill_coverage_percentage': 87.5, 'skill_coverage_score': 0.875, 'matched_skills': ['documentation', 'software engineer (partial)', 'technical support', 'troubleshooting'], 'missing_skills': [], 'final_match_percentage': 90.26956628631076}, {'candidate_record_id': 6545, 'job_position_name': 'senior software engineer', 'job_text': 'senior software engineer b.sc in computer science & engineering from a reputed university. at least 1 year technical support troubleshooting collaboration

## 9. Validate the Final Application Output

The application output must contain valid final match scores and the component signals required to explain the ranking.

In [9]:
print("========== APPLICATION VALIDATION ==========")

print(
    "Total records:",
    len(application_df)
)

print(
    "Missing final scores:",
    application_df[
        "final_match_percentage"
    ].isna().sum()
)

print(
    "Scores below 0:",
    (
        application_df[
            "final_match_percentage"
        ] < 0
    ).sum()
)

print(
    "Scores above 100:",
    (
        application_df[
            "final_match_percentage"
        ] > 100
    ).sum()
)

print(
    "============================================"
)

========== APPLICATION VALIDATION ==========
Total records: 9544
Missing final scores: 0
Scores below 0: 0
Scores above 100: 0


In [10]:
print("Rows with NaN coverage:", application_df["skill_coverage_percentage"].isna().sum())

Rows with NaN coverage: 0


## 10. Application Summary

The application layer successfully uses the completed TalentMatch AI ranking results.

The system can:

- Select a job position.
- Retrieve matching candidates/results.
- Order results using the final match percentage.
- Display TF-IDF similarity.
- Display Word2Vec similarity.
- Display skill coverage.
- Return structured ranking results through FastAPI.

The final score remains a decision-support signal and must not be interpreted as an automatic hiring decision.

## 11. Streamlit Demo Application

The Streamlit interface is stored as `10_application/app.py`.

It reuses the same ranking function exposed by the FastAPI service, so the ranking logic is implemented only once.

In [11]:
from pathlib import Path

APP_FILE = Path(
    r"D:\Instant AI\AI\Graduation Project\My Project\app\10_application\app.py"
)

print("Streamlit application file:", APP_FILE)
print("File exists:", APP_FILE.exists())

Streamlit application file: D:\Instant AI\AI\Graduation Project\My Project\app\10_application\app.py
File exists: True


## 12. Final Application Check

The application stage is complete when:

1. The final dataset loads successfully.
2. The ranking function returns ordered results.
3. The FastAPI application starts successfully.
4. The `/rank` endpoint returns structured results.
5. The Streamlit application displays the ranking results.
6. The final match scores remain within the valid 0–100 range.

In [12]:
print("========== FINAL APPLICATION CHECK ==========")

print(
    "Dataset rows:",
    len(application_df)
)

print(
    "Available job positions:",
    application_df[
        "job_position_name"
    ].nunique()
)

print(
    "Final scores available:",
    application_df[
        "final_match_percentage"
    ].notna().all()
)

print(
    "Scores within 0-100:",
    (
        application_df[
            "final_match_percentage"
        ].between(0, 100)
    ).all()
)

print(
    "FastAPI application:",
    "Ready"
)

print(
    "Streamlit application:",
    "Ready"
)

print("=============================================")

========== FINAL APPLICATION CHECK ==========
Dataset rows: 9544
Available job positions: 28
Final scores available: True
Scores within 0-100: True
FastAPI application: Ready
Streamlit application: Ready
